# N090 · 欧拉路径与Hierholzer

**目标：** 在边不重用条件下构造完整行程。

**先修：** N051, N079, N080, N037。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 边与点遍历区别；度条件；连通性；后序构造；词典序；重复边。

## 从问题到算法

欧拉路径要求每条边恰好使用一次，而不是每个顶点一次。沿未用边一直走，无路可走时才把顶点加入逆序答案，相当于把局部回路插入完整行程。机票使用多重边；重复机票不能去重。字典序要求对出边排序，终局还必须检查确实消耗了全部机票。

## 最小实现

**本章接口：** `eulerian_path(edges)`、`find_itinerary(tickets)`

In [1]:
from collections import defaultdict,Counter

def _hierholzer(edges,start):
    adj=defaultdict(list)
    for u,v in edges: adj[u].append(v)
    for a in adj.values(): a.sort(reverse=True)
    stack=[start]; out=[]
    while stack:
        if adj[stack[-1]]: stack.append(adj[stack[-1]].pop())
        else: out.append(stack.pop())
    path=out[::-1]
    if len(path)!=len(edges)+1 or Counter(zip(path,path[1:]))!=Counter(map(tuple,edges)): return None
    return path

def eulerian_path(edges):
    if not edges: return []
    delta=Counter()
    for u,v in edges: delta[u]+=1; delta[v]-=1
    starts=[u for u,d in delta.items() if d==1]; ends=[u for u,d in delta.items() if d==-1]
    if any(abs(d)>1 for d in delta.values()) or not (len(starts)==len(ends)==1 or not starts and not ends): return None
    start=starts[0] if starts else min(u for u,v in edges)
    return _hierholzer(edges,start)

def find_itinerary(tickets):
    path=_hierholzer(tickets,'JFK')
    if path is None: raise ValueError('no itinerary using all tickets from JFK')
    return path

## 正确性、前提与复杂度

逆序退出将闭合支路拼到恰当顶点；出边每弹出一次只消耗该条边。可行欧拉条件下，按字典序尝试的闭合支路可以先拼接，无法提前闭合的终止支路被留到末尾，得到字典序最小的完整行程。最终Counter校验排除不连通和非法拼接。

**代价：** 排序出边O(E log E)，遍历O(E)，辅助O(V+E)。通用接口要求顶点标签可排序且可哈希；返回None表示无完整欧拉路径。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

tickets=[('JFK','KUL'),('JFK','NRT'),('NRT','JFK')]; path=find_itinerary(tickets)
show_table(['步骤','起点','终点'],[(i,u,v) for i,(u,v) in enumerate(zip(path,path[1:]),1)])

步骤,起点,终点
1,JFK,NRT
2,NRT,JFK
3,JFK,KUL


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert find_itinerary([('JFK','KUL'),('JFK','NRT'),('NRT','JFK')])==['JFK','NRT','JFK','KUL']
assert eulerian_path([(0,1),(0,1),(1,0)])==[0,1,0,1]
assert eulerian_path([(0,1),(2,3)]) is None
assert eulerian_path([])==[]
from itertools import permutations
edges=[('JFK','A'),('A','JFK'),('JFK','B'),('B','JFK'),('JFK','A')]
valid=[]
for perm in permutations(edges):
    path=['JFK']
    for u,v in perm:
        if path[-1]!=u: break
        path.append(v)
    if len(path)==len(edges)+1: valid.append(path)
assert find_itinerary(edges)==min(valid)
print('N090: 所有本章断言通过')

N090: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 构造逐步选最小边但不回溯会卡住的例子。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 检查重票按多重边消费。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 332. Reconstruct Itinerary（canonical）
- 2097. Valid Arrangement of Pairs（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。